In [1]:
from lightcurves import LC_DATA_PATH

from tqdm.notebook import tqdm

from pathlib import Path

In [2]:
MODEL_SAVE_PATH = Path('data/cnn/model_weights.pth')
MODEL_SAVE_PATH.parent.mkdir(exist_ok=True, parents=True)
STOPPING_PATIENCE = 5
STOPPING_SIGMA = 0.1

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR

import numpy

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
print('Device:', device)
# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resizing the images to fit ResNet input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset
train_dataset = torchvision.datasets.ImageFolder(root=LC_DATA_PATH, transform=transform)

# Split the dataset into training and testing sets (80% train, 20% test)
train_size = int(0.8 * len(train_dataset))
test_size = len(train_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(train_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Define the model
model = models.resnet34(pretrained=True) # Load the pre-trained ResNet model
model.fc = nn.Linear(model.fc.in_features, 2)  # Change the output layer to have two classes
model = torch.nn.DataParallel(model.to(device))

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

accuracies = []

# Training loop
epoch = 0
while True:
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc='Train', leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

    for inputs, labels in tqdm(test_loader, desc='Test', leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()
    
    # Print statistics
    accuracies.append(100 * correct / total)
    accuracies = accuracies[-STOPPING_PATIENCE:]
    accuracy_sigma = numpy.std(accuracies)
    print(f"Epoch [{epoch+1}], Loss: {running_loss:.4f}, Accuracy: {accuracies[-1]:.2f}%, Accuracy std: {accuracy_sigma:.2f}")
    if len(accuracies) >= STOPPING_PATIENCE and accuracy_sigma <= STOPPING_SIGMA:
        break
    epoch += 1


print("Training completed!")

Device: mps


/Users/adam/Code/lc-filter/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/adam/Code/lc-filter/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Train:   0%|          | 0/500 [00:00<?, ?it/s]

Test:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch [1], Loss: 26.9768, Accuracy: 92.75%, Accuracy std: 0.00


...

Test:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch [26], Loss: 93.0523, Accuracy: 93.25%, Accuracy std: 0.18


Train:   0%|          | 0/500 [00:00<?, ?it/s]

Test:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch [27], Loss: 95.6760, Accuracy: 93.17%, Accuracy std: 0.08
Training completed!


In [4]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)